# Tinh chỉnh phần đầu (classifier finetuning)

Mục tiêu:
- Khởi tạo lại kiến trúc VGG16 và đóng băng phần thân (kế thừa từ Notebook 2).
- Gỡ bỏ lớp phân loại 1000 đồ vật (ImageNet) mặc định của VGG16.
- Thay thế bằng một lớp phân loại (Dense/Linear) mới gồm 10 ngõ ra tương ứng với 10 loại quần áo của tập FashionMNIST.
- Thống kê số lượng tham số (Parameters) để chứng minh tính hiệu quả của kỹ thuật Transfer Learning.

In [1]:
import torch
import torch.nn as nn
from torchvision import models
import os

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Sử dụng thiết bị CUDA: {device}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Sử dụng thiết bị Apple Metal Performance Shaders: {device}")
else:
    device = torch.device("cpu")
    print(f"Sử dụng CPU: {device}")

model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
for param in model.features.parameters():
    param.requires_grad = False

Sử dụng thiết bị Apple Metal Performance Shaders: mps


## Thay thế lớp phân loại (Gắn đầu mới)
Trong kiến trúc VGG16 của PyTorch, phần `classifier` là một chuỗi (Sequential) gồm 7 bước (đánh số từ 0 đến 6).

Lớp cuối cùng là `classifier[6]`, một lớp Tuyến tính (Linear) nhận đầu vào là 4096 đặc trưng và xuất ra 1000 nhãn.

Ta sẽ thay thế riêng lớp số `[6]` này thành một lớp Linear mới xuất ra 10 nhãn (số class của FashionMNIST).

In [5]:
print("Cấu trúc classifier trước khi chỉnh sửa")
print(model.classifier)

# Xác định số lượng nhãn của bài toán
num_classes = 10

# Lấy số lượng nút đầu vào (in_features) của lớp classifier số 6 gốc (thường là 4096 đối với VGG16)
in_features = model.classifier[6].in_features

# Ghi đè lớp cuối cùng bằng một lớp hoàn toàn mới
model.classifier[6] = nn.Linear(in_features, num_classes)

# Chuyển toàn bộ mô hình lên thiết bị train
model = model.to(device)

print("Cấu trúc classifier sau khi chỉnh sửa")
print(model.classifier)

Cấu trúc classifier trước khi chỉnh sửa
Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=10, bias=True)
)
Cấu trúc classifier sau khi chỉnh sửa
Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=10, bias=True)
)


## Thống kê Tham số tính toán
Khác với mô hình CNN tự xây (Custom CNN) phải tính đạo hàm cho 100% tham số, ở đây ta sẽ dùng code để chứng minh sự tối ưu của Transfer Learning. Ta sẽ đếm tổng số tham số của mô hình và số lượng tham số thực sự cần phải học (những tham số có `requires_grad = True`).

In [3]:
# Tính tổng toàn bộ tham số trong mạng VGG16
total_params = sum(p.numel() for p in model.parameters())

# Tính số tham số thực sự cần cập nhật trong lúc train (chỉ nằm ở phần Classifier)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Tổng số tham số của mô hình (Total Params): {total_params:,}")
print(f"Số tham số cần huấn luyện (Trainable Params): {trainable_params:,}")
print(f"Tỷ lệ tham số phải tính toán đạo hàm: {(trainable_params/total_params)*100:.4f}%")

Tổng số tham số của mô hình (Total Params): 134,301,514
Số tham số cần huấn luyện (Trainable Params): 119,586,826
Tỷ lệ tham số phải tính toán đạo hàm: 89.0435%


## Lưu lại cấu trúc ban đầu
Để việc huấn luyện chạy mượt mà mà không cần định nghĩa lại từ đầu, ta sẽ lưu lại trạng thái khởi tạo này của mô hình.

In [4]:
# Tạo thư mục models nếu chưa có
os.makedirs("models", exist_ok=True)

# Lưu lại trọng số (state_dict) ban đầu
save_path = "models/vgg16_untrained_custom_head.pth"
torch.save(model.state_dict(), save_path)

print(f"Đã lưu thành công cấu trúc VGG16 tùy chỉnh vào: {save_path}")

Đã lưu thành công cấu trúc VGG16 tùy chỉnh vào: models/vgg16_untrained_custom_head.pth
